## Preprocesamiento del Corpus

In [1]:
import os
import re
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
#paralelización
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor


In [6]:
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords as nltk_sw

In [3]:
# ── Configuración global ──────────────────────────────────────────────────────
PATH    = r'C:\\Users\\luis3\\Downloads\\RI26A\\Recuperacion_Informacion\\data\\gutenberg_top100'
WORKERS = os.cpu_count()   # todos los núcleos disponibles
##STOP    = set(nltk_sw.words('english'))
print(f'Núcleos disponibles: {WORKERS}')

# ── Carga paralela de archivos (I/O bound → ThreadPoolExecutor) ───────────────
def leer_archivo(args):
    path, archivo = args
    with open(os.path.join(path, archivo), 'r', encoding='utf-8', errors='ignore') as f:
        return archivo, f.read()
# Listar archivos y preparar argumentos para la función de lectura
files = [f for f in os.listdir(PATH) if f.endswith('.txt')]
args  = [(PATH, f) for f in files]
# Leer archivos en paralelo
with ThreadPoolExecutor(max_workers=WORKERS) as pool:
    resultados = list(pool.map(leer_archivo, args))

titulos = [r[0] for r in resultados]
corpus  = [r[1] for r in resultados]

print(f'Corpus cargado: {len(corpus)} documentos')

Núcleos disponibles: 4
Corpus cargado: 103 documentos


## Limpieza de datos

In [4]:
# ── Limpieza de datos con expresiones regulares ──────────────────────────────

def limpiar_texto(texto):
    # Eliminar Project Gutenberg headers/footers
    texto = re.sub(r'\*\*\* START OF .*? \*\*\*', '', texto)
    texto = re.sub(r'\*\*\* END OF .*? \*\*\*', '', texto)
    
    # Eliminar caracteres especiales y anotaciones
    texto = re.sub(r'\[.*?\]', '', texto)  # Eliminar [Illustration], etc.
    texto = re.sub(r'\{.*?\}', '', texto)  # Eliminar anotaciones entre {}
    
    # Convertir a minúsculas
    texto = texto.lower()
    
    # Eliminar URLs
    texto = re.sub(r'http\S+|www\S+', '', texto)
    
    # Eliminar puntuación y caracteres especiales (excepto espacios)
    texto = re.sub(r'[^a-z0-9\s]', '', texto)
    
    # Eliminar espacios múltiples
    texto = re.sub(r'\s+', ' ', texto)
    
    # Eliminar espacios al inicio y final
    texto = texto.strip()
    
    return texto

# Aplicar limpieza a todos los documentos
corpus_limpio = [limpiar_texto(doc) for doc in corpus]

print(f'Corpus limpiado: {len(corpus_limpio)} documentos')
print(f'Longitud promedio: {np.mean([len(doc) for doc in corpus_limpio]):.0f} caracteres')

Corpus limpiado: 103 documentos
Longitud promedio: 830080 caracteres


## Tocken

In [7]:
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()#stemmer es un objeto que se utiliza para 
                        #realizar el proceso de stemming, 
                        # que consiste en reducir las palabras 
                        # a su raíz o forma base. 
                        # En este caso, se está utilizando 
                        # el Porter Stemmer, que es uno de 
                        # los algoritmos de stemming más 
                        # comunes y efectivos para el idioma inglés.

def stem_texto(texto):
    tokens = texto.split()
    stems = [stemmer.stem(token) for token in tokens]
    return " ".join(stems)

corpus_stem = [stem_texto(doc) for doc in corpus_limpio]

print(f'Corpus con stemming: {len(corpus_stem)} documentos')
print(corpus_stem[0][:500])

Corpus con stemming: 103 documentos
the complet work of william shakespear by william shakespear content the sonnet all well that end well the tragedi of antoni and cleopatra as you like it the comedi of error the tragedi of coriolanu cymbelin the tragedi of hamlet princ of denmark the first part of king henri the fourth the second part of king henri the fourth the life of king henri the fifth the first part of henri the sixth the second part of king henri the sixth the third part of king henri the sixth king henri the eighth the 


In [8]:
vectorizador = TfidfVectorizer(stop_words="english")
matriz_tfidf = vectorizador.fit_transform(corpus_stem)

In [9]:
print (f'Matriz TF-IDF: {matriz_tfidf.shape[0]} documentos, {matriz_tfidf.shape[1]} términos')

Matriz TF-IDF: 103 documentos, 159311 términos


In [ ]:
def procesador_texto(texto):
    texto_limpio = limpiar_texto(texto)
    return stem_texto(texto_limpio)

resultado = [procesador_texto(doc) for doc in corpus]

df_corpus = pd.DataFrame({
    "raw": corpus,
    "procesado": resultado,
    "titulo": titulos
})

vectorizador = TfidfVectorizer(stop_words="english")
matriz_tfidf = vectorizador.fit_transform(df_corpus["procesado"])

querry = "What is the meaning of life?"
querry_procesada = procesador_texto(querry)


df_corpus[["titulo", "procesado"]].head()

,titulo,procesado
0,100_Complete_Works_Shakespeare.txt,the complet work of william shakespear by will...
1,1023_Bleak_House.txt,bleak hous by charl dicken content prefac i in...
2,1080_A_Modest_Proposal.txt,a modest propos for prevent the children of po...
3,1184_The_Count_of_Monte_Cristo.txt,the count of mont cristo by alexandr duma cont...
4,11_Alices_Adventures_in_Wonderland.txt,alic adventur in wonderland by lewi carrol the...


In [13]:
query_tfidf = vectorizador.transform([querry_procesada])
similaridades = cosine_similarity(query_tfidf, matriz_tfidf).flatten()
df_corpus["similaridad"] = similaridades
df_corpus.sort_values("similaridad", ascending=False)[["titulo", "similaridad"]].head(10)



,titulo,similaridad
16,1497_The_Republic.txt,0.114307
10,1322_Leaves_of_Grass.txt,0.103747
29,205_Walden.txt,0.090532
58,4363_Beyond_Good_and_Evil.txt,0.090368
56,408_The_Souls_of_Black_Folk.txt,0.084674
60,45304_City_of_God_Vol1.txt,0.077701
25,174_Picture_of_Dorian_Gray.txt,0.070348
42,2680_Meditations.txt,0.063766
97,pg42538.txt,0.062889
102,pg84.txt,0.062856
